# 05 Data Quality and Expectations

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Design data quality checks for pipelines: write rules as data, choose between <b>warn</b>, <b>drop</b> and <b>fail</b> for each rule, route bad records to a <b>quarantine</b> table with the reasons they failed, measure quality per batch, and implement the same design in a Lakeflow Declarative Pipeline with the Python expectations API.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Bad data that silently reaches a dashboard costs more than a failed job. But failing on every bad record stops the business too. Good pipelines decide per rule: some issues are fatal, some records should be set aside for review, some only need monitoring. The exam asks which expectation action fits a scenario, and interviews ask "how do you handle bad data?".
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Choosing an action per rule</b><br>
| Action | Behavior | Use when |<br>|---|---|---|<br>| <b>Warn</b> (<code>expect</code>) | Keep the record, count the violation | Suspicious but usable data, or a new rule you're still evaluating |<br>| <b>Drop</b> (<code>expect_or_drop</code>) | Remove the record, count it | The record is unusable, and losing it is acceptable |<br>| <b>Fail</b> (<code>expect_or_fail</code>) | Stop the update | The data would corrupt results, e.g. a broken schema or duplicate primary keys |<br>| <b>Quarantine</b> | Send invalid records to a separate table with reasons | Records must be reviewed, fixed and replayed, not lost |<br><br>
Rules belong in one place (a dictionary or a table), so the same definitions drive filtering, quarantine and reporting.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An insurer's claims pipeline dropped every record with a missing policy number, about 0.3% of claims. Months later, finance found millions in claims missing from reports. They switched to a quarantine: invalid claims went to <code>claims_quarantine</code> with the failed rule names, a daily alert reported the count, and an operations team fixed and replayed them. Rules that indicated a broken feed (more than 5% invalid) were made fatal, so a bad upstream release now stops the pipeline instead of silently dropping data.
</div>

## Syntax

```python
from pyspark import pipelines as dp          # or: import dlt as dp

RULES = {"valid_amount": "amount > 0", "has_customer": "customer_id IS NOT NULL"}

@dp.table
@dp.expect_all_or_drop(RULES)                 # also: expect_all (warn), expect_all_or_fail
def orders_clean():
    return spark.readStream.table("orders_bronze")

@dp.table
def orders_quarantine():
    bad = " OR ".join(f"NOT coalesce(({r}), false)" for r in RULES.values())
    return spark.readStream.table("orders_bronze").where(bad)
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and a batch of incoming orders with a variety of problems.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
10 orders.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
BASE = f"/Volumes/{CATALOG}/workflows/files"

In [0]:
from pyspark.sql import functions as F

incoming = spark.createDataFrame([
    (1, 101, 25.0, "INR", "2024-11-01"), (2, 102, 40.0, "GBP", "2024-11-01"), (3, 101, -5.0, "INR", "2024-11-01"),
    (4, None, 12.5, "AED", "2024-11-02"), (5, 103, 60.0, "XYZ", "2024-11-02"), (6, 102, None, "GBP", "2024-11-02"),
    (7, 104, 99999.0, "INR", "2024-11-03"), (8, 105, 30.0, "AED", "1999-01-01"), (9, 106, 18.0, "GBP", "2024-11-03"),
    (10, 107, 22.0, "INR", "2024-11-03"),
], "order_id INT, customer_id INT, amount DOUBLE, currency STRING, order_date STRING").withColumn("order_date", F.to_date("order_date"))
print("incoming:", incoming.count())

## 1. Rules as data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines each rule with a SQL condition and an action, and evaluates every rule on every row, collecting the names of the failed rules in an array.<br><br>
<b>Why it matters</b><br>One list of rules drives everything that follows. The <code>failed_rules</code> array explains <b>why</b> a record is bad, which is what people fixing data need. A <code>NULL</code> result counts as a failure, so a missing amount doesn't slip through <code>amount &gt; 0</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each order with the list of rules it fails. Orders 1, 2, 9 and 10 fail nothing.
</div>

In [0]:
RULES = [
    # name,              condition,                                     action
    ("has_amount",       "amount IS NOT NULL",                          "drop"),
    ("valid_amount",     "amount > 0",                                  "quarantine"),
    ("has_customer",     "customer_id IS NOT NULL",                     "quarantine"),
    ("known_currency",   "currency IN ('INR', 'GBP', 'AED', 'USD')",    "quarantine"),
    ("plausible_amount", "amount < 10000",                              "warn"),
    ("recent_date",      "order_date >= DATE'2020-01-01'",              "warn"),
]

def with_failed_rules(df, rules):
    checks = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(name)) for name, cond, _ in rules]
    return df.withColumn("failed_rules", F.filter(F.array(*checks), lambda x: x.isNotNull()))

checked = with_failed_rules(incoming, RULES)
checked.select("order_id", "amount", "currency", "failed_rules").orderBy("order_id").show(truncate=False)

## 2. Apply the actions: clean, quarantine, dropped

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Splits the batch by the <b>strictest</b> action among each record's failed rules: records failing a <code>drop</code> rule are discarded, records failing a <code>quarantine</code> rule go to the quarantine table with their reasons, and everything else (including records with only warnings) goes to the clean table.<br><br>
<b>Why it matters</b><br>This is the full pattern: nothing is lost silently except what you've explicitly decided to drop, and warnings are recorded without blocking.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Clean: 6 orders (1, 2, 9, 10, plus 7 and 8, which only have warnings). Quarantine: 3 orders (3, 4, 5) with their reasons. Dropped: 1 (order 6, which has no amount).
</div>

In [0]:
RANK = {"warn": 1, "quarantine": 2, "drop": 3}
rank_of_rule = F.create_map(*[F.lit(x) for name, _, action in RULES for x in (name, RANK[action])])
worst = F.aggregate(F.transform("failed_rules", lambda r: rank_of_rule[r]), F.lit(0), lambda acc, x: F.greatest(acc, x))
routed = checked.withColumn("worst", worst)

clean = routed.filter("worst <= 1").drop("worst")
quarantine = routed.filter("worst = 2").drop("worst").withColumn("quarantined_at", F.current_timestamp())
dropped = routed.filter("worst = 3")

clean.write.mode("overwrite").saveAsTable(f"{DB}.dq_orders_clean")
quarantine.write.mode("overwrite").saveAsTable(f"{DB}.dq_orders_quarantine")
print("clean:", clean.count(), "| quarantine:", quarantine.count(), "| dropped:", dropped.count())
spark.table(f"{DB}.dq_orders_quarantine").select("order_id", "failed_rules").orderBy("order_id").show(truncate=False)

## 3. Quality metrics per batch

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts the violations of each rule in the batch and computes the failure rate, then appends them to a metrics table with a batch ID.<br><br>
<b>Why it matters</b><br>Trends matter more than single numbers: 0.1% invalid every day is normal, 8% today means something broke upstream. A metrics table is what dashboards and alerts are built on.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per rule with its violation count and rate. <code>valid_amount</code> has 2 violations (20%): the negative amount, and the missing amount, because <code>NULL</code> counts as a failure.
</div>

In [0]:
import uuid
batch_id = str(uuid.uuid4())[:8]
total = incoming.count()
metrics = (checked.select(F.explode("failed_rules").alias("rule")).groupBy("rule").count()
           .join(spark.createDataFrame([(n, a) for n, _, a in RULES], "rule STRING, action STRING"), "rule", "right")
           .fillna(0, ["count"])
           .select(F.lit(batch_id).alias("batch_id"), F.current_timestamp().alias("checked_at"), "rule", "action",
                   F.col("count").alias("violations"), F.round(F.col("count") * 100 / total, 1).alias("pct")))
metrics.write.mode("append").saveAsTable(f"{DB}.dq_metrics")
spark.table(f"{DB}.dq_metrics").filter(F.col("batch_id") == batch_id).orderBy("rule").select("rule", "action", "violations", "pct").show()

## 4. Fatal thresholds

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a guard: if more than 30% of a batch is quarantined or dropped, raise an error instead of loading it.<br><br>
<b>Why it matters</b><br>A few bad records are normal. A large share usually means a broken feed (a changed format, a wrong file), and loading 70% of the data silently is worse than stopping. This is the "fail" action at the batch level.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
40% of this batch is invalid, so the check raises an error, which the cell catches and prints.
</div>

In [0]:
def check_threshold(total, bad, max_bad_pct=30):
    pct = bad * 100 / total
    if pct > max_bad_pct:
        raise ValueError(f"{pct:.0f}% of the batch is invalid (limit {max_bad_pct}%): stopping the load")
    return f"{pct:.0f}% invalid, within the limit"

try:
    print(check_threshold(total, quarantine.count() + dropped.count()))
except ValueError as e:
    print("FAILED:", e)

## 5. The same design in a declarative pipeline

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a Python pipeline with Auto Loader for Bronze, <code>expect_all_or_drop</code> for the clean table, <code>expect_all</code> (warn) rules, and a quarantine table. Runs it and reads the counts.<br><br>
<b>Why it matters</b><br>In Lakeflow Declarative Pipelines, the actions are decorators and the metrics are recorded for you. The quarantine is a second table with the inverse condition, built from the same rule dictionary.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Runs as a serverless pipeline in Free Edition, which takes a few minutes. If pipeline creation fails, the cell prints the error.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>COMPLETED</code>, then Bronze 10 rows, clean 6 and quarantine 4. The clean table excludes the 4 records failing the drop rules (orders 3, 4, 5, 6), and the quarantine table contains them.
</div>

In [0]:
import base64, time, os, shutil
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import pipelines, workspace

w = WorkspaceClient()
me = w.current_user.me().user_name
FOLDER = f"/Users/{me}/databricks-learning-workflows"
w.workspace.mkdirs(FOLDER)

RAW = f"{BASE}/dq_orders"
shutil.rmtree(RAW, ignore_errors=True)
incoming.write.mode("overwrite").option("header", True).csv(RAW)

DROP_RULES = {n: c for n, c, a in RULES if a in ("drop", "quarantine")}
WARN_RULES = {n: c for n, c, a in RULES if a == "warn"}
PY_PIPELINE = f'''
try:
    from pyspark import pipelines as dp
except ImportError:
    import dlt as dp

DROP_RULES = {DROP_RULES!r}
WARN_RULES = {WARN_RULES!r}

@dp.table(comment="Raw orders loaded with Auto Loader")
def dq_pipe_bronze():
    return (spark.readStream.format("cloudFiles").option("cloudFiles.format", "csv").option("header", "true")
            .schema("order_id INT, customer_id INT, amount DOUBLE, currency STRING, order_date DATE")
            .load("{RAW}"))

@dp.table(comment="Orders passing all quality rules")
@dp.expect_all_or_drop(DROP_RULES)
@dp.expect_all(WARN_RULES)
def dq_pipe_clean():
    return spark.readStream.table("dq_pipe_bronze")

@dp.table(comment="Orders failing a drop rule, kept for review")
def dq_pipe_quarantine():
    bad = " OR ".join(f"NOT coalesce(({{c}}), false)" for c in DROP_RULES.values())
    return spark.readStream.table("dq_pipe_bronze").where(bad)
'''
py_path = f"{FOLDER}/dq_orders_pipeline"
w.workspace.import_(path=py_path, format=workspace.ImportFormat.SOURCE, language=workspace.Language.PYTHON,
                    content=base64.b64encode(PY_PIPELINE.encode()).decode(), overwrite=True)

for old in w.pipelines.list_pipelines(filter="name LIKE 'dbl_05_dq_pipeline'"):
    w.pipelines.delete(pipeline_id=old.pipeline_id)
settings = dict(name="dbl_05_dq_pipeline", catalog=CATALOG, serverless=True, development=True, continuous=False,
                libraries=[pipelines.PipelineLibrary(notebook=pipelines.NotebookLibrary(path=py_path))])
try:
    PIPELINE_ID = w.pipelines.create(schema="workflows", **settings).pipeline_id
except TypeError:
    PIPELINE_ID = w.pipelines.create(target="workflows", **settings).pipeline_id

update_id = w.pipelines.start_update(pipeline_id=PIPELINE_ID).update_id
while (st := w.pipelines.get_update(pipeline_id=PIPELINE_ID, update_id=update_id).update.state.value) not in ("COMPLETED", "FAILED", "CANCELED"):
    time.sleep(15)
print("update:", st)
for t in ["dq_pipe_bronze", "dq_pipe_clean", "dq_pipe_quarantine"]:
    print(f"{t:<20} {spark.table(f'{DB}.{t}').count()} rows")

## 6. Read the pipeline's quality metrics, then clean up

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the expectation metrics from the event log, then deletes the pipeline (and its tables) unless <code>KEEP_PIPELINE = True</code>.<br><br>
<b>Why it matters</b><br>The same numbers you computed by hand in section 3 come for free from a declarative pipeline.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Passed and failed counts per rule for <code>dq_pipe_clean</code>, then <code>pipeline deleted</code>.
</div>

In [0]:
display(spark.sql(f"""
    SELECT e.dataset, e.name, SUM(e.passed_records) AS passed, SUM(e.failed_records) AS failed
    FROM (SELECT explode(from_json(details:flow_progress:data_quality:expectations,
                 'array<struct<name: string, dataset: string, passed_records: bigint, failed_records: bigint>>')) AS e
          FROM event_log('{PIPELINE_ID}')
          WHERE event_type = 'flow_progress' AND details:flow_progress:data_quality:expectations IS NOT NULL)
    GROUP BY ALL ORDER BY 1, 2
"""))
KEEP_PIPELINE = False
if not KEEP_PIPELINE:
    w.pipelines.delete(pipeline_id=PIPELINE_ID)
    print("pipeline deleted")

## Under the hood

```
record ─▶ evaluate every rule (NULL = fail) ─▶ failed_rules = [..]
   ├─ worst action = drop        ─▶ discarded (counted)
   ├─ worst action = quarantine  ─▶ quarantine table (+ reasons, timestamp) ─▶ review, fix, replay
   └─ none or warn only          ─▶ clean table (warnings counted)
batch: bad share > threshold ─▶ fail the load (broken feed)
metrics per rule per batch ─▶ metrics table / event log ─▶ dashboard + alerts
```

In [0]:
spark.table(f"{DB}.dq_orders_clean").select("order_id", "failed_rules").orderBy("order_id").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: rules that let NULL through</b><br>
<code>amount &gt; 0</code> is <code>NULL</code> for a missing amount, and a filter on it drops the row while a check on <code>NOT (amount &gt; 0)</code> keeps it. Wrap conditions with <code>coalesce(cond, false)</code> and add explicit <code>IS NOT NULL</code> rules.<br><br>
<b>⛔ Problem: dropping records that someone needs</b><br>Use a quarantine with reasons instead, and report its size.<br><br>
<b>⛔ Problem: failing the whole pipeline on one bad record</b><br>Reserve fail for rules that indicate broken data. Use thresholds for "too many bad records".<br><br>
<b>⛔ Problem: rules duplicated in several notebooks</b><br>Keep them in one dictionary or table, and generate checks from it.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you handle bad records in a pipeline?</b><br>
Define rules centrally, then decide per rule: warn for suspicious but usable data, drop for unusable records we can lose, quarantine with reasons for records that must be reviewed and replayed, and fail for issues that would corrupt results. I also track violation counts per batch and fail the load if the bad share exceeds a threshold, because that usually means a broken feed.<br><br>

<b>🎤 2. How do you implement a quarantine table in Lakeflow Declarative Pipelines?</b><br>
Apply <code>expect_all_or_drop</code> with the rules on the clean table, and create a second table that reads the same source with the inverted condition (any rule failing, treating NULL as a failure), so invalid records are kept with their data for review.<br><br>

<b>🎤 3. What's the difference between expectations and Delta <code>CHECK</code> constraints?</b><br>
A <code>CHECK</code> constraint rejects the whole write if any row violates it, on any writer. Expectations work inside pipelines and can warn, drop rows or fail the update, with metrics recorded per rule.<br><br>

<b>🎤 4. How do you monitor data quality over time?</b><br>
Store per-batch, per-rule violation counts (from the pipeline event log or a metrics table), chart them in a dashboard, and alert on spikes relative to the usual rate.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Records with an invalid <code>country_code</code> should be removed from the target table, and the pipeline should keep running. Which expectation fits?</b><br>
A. <code>@dp.expect("valid_country", "country_code IS NOT NULL")</code><br>
B. <code>@dp.expect_or_drop("valid_country", "country_code IN (...)")</code><br>
C. <code>@dp.expect_or_fail("valid_country", "country_code IN (...)")</code><br>
D. A <code>CHECK</code> constraint on the target table<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A data engineer wants to keep all records but track how many violate a new rule before enforcing it. Which action should they use?</b><br>
A. Warn (<code>expect</code>)<br>
B. Drop (<code>expect_or_drop</code>)<br>
C. Fail (<code>expect_or_fail</code>)<br>
D. <code>VACUUM</code><br>
<details><summary><b>Show answer</b></summary><b>A.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a rule <code>positive_customer</code> (<code>customer_id &gt; 0</code>) with action <code>quarantine</code>, and re-run sections 1–2</li><li>Change the threshold check so that <code>drop</code> records count double (they're lost forever)</li><li>Write a query that shows, from <code>dq_metrics</code>, the rules whose violation rate in the latest batch is above 5%</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Extra rule
rules2 = RULES + [("positive_customer", "customer_id > 0", "quarantine")]
with_failed_rules(incoming, rules2).filter(F.array_contains("failed_rules", "positive_customer")).select("order_id", "failed_rules").show(truncate=False)

# 2. Weighted threshold
def check_weighted(total, quarantined, dropped_n, max_bad_pct=30):
    pct = (quarantined + 2 * dropped_n) * 100 / total
    return ("FAIL" if pct > max_bad_pct else "OK", f"{pct:.0f}%")
print(check_weighted(total, quarantine.count(), dropped.count()))

# 3. Rules above 5% in the latest batch
latest = spark.table(f"{DB}.dq_metrics").orderBy(F.desc("checked_at")).first()["batch_id"]
spark.table(f"{DB}.dq_metrics").filter((F.col("batch_id") == latest) & (F.col("pct") > 5)).select("rule", "violations", "pct").orderBy("rule").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Rules as data, with an action each: warn, drop, quarantine, fail</li><li>Treat <code>NULL</code> results as failures (<code>coalesce(cond, false)</code>)</li><li>Quarantine keeps bad records with reasons for review and replay</li><li>Track violations per rule per batch, and fail on abnormal bad shares</li><li>In declarative pipelines: <code>expect</code>, <code>expect_or_drop</code>, <code>expect_or_fail</code> (and <code>_all</code> variants), with metrics in the event log</li></ul>
</div>

| Python | SQL | Effect |
|---|---|---|
| `@dp.expect(name, cond)` | `CONSTRAINT n EXPECT (cond)` | Warn |
| `@dp.expect_or_drop` | `... ON VIOLATION DROP ROW` | Drop row |
| `@dp.expect_or_fail` | `... ON VIOLATION FAIL UPDATE` | Fail update |
| `@dp.expect_all*(dict)` | several constraints | Many rules at once |

## Git commit

```
git add 07_workflows/05_data_quality_expectations.ipynb
git commit -m "add 07_05 data quality expectations notebook"
```